# House Price Prediction Project

# Load Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import  statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# import function to perform linear regression
from sklearn.linear_model import LinearRegression

from sklearn.model_selection import KFold, cross_val_score

# import StandardScaler to perform scaling
from sklearn.preprocessing import StandardScaler 

# import SGDRegressor from sklearn to perform linear regression with stochastic gradient descent
from sklearn.linear_model import SGDRegressor


# import function for ridge regression
from sklearn.linear_model import Ridge

# import function for lasso regression
from sklearn.linear_model import Lasso

# import function for elastic net regression
from sklearn.linear_model import ElasticNet

# import function to perform GridSearchCV
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings('ignore')

# Load Dataset

In [ ]:
df = pd.read_csv("02_housing.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

# Checking for Duplicates

In [ ]:
df.duplicated().sum()

# Handling Missing Values

In [ ]:
# display the total number of null values in each column of dataframe

# 'ascending = False' sorts values in the descending order
Total = df.isnull().sum().sort_values(ascending=False)          

# percentage of missing values
Percent = (df.isnull().sum()*100/df.isnull().count()).sort_values(ascending=False)  

# concat the 'Total' and 'Percent' columns using 'concat' function
# 'keys' is the list of column names
# 'axis = 1' concats along the columns
missing_data = pd.concat([Total, Percent], axis=1, keys=['Total', 'Percent'])    
missing_data

In [ ]:
df_missing_values = df[['CRIM','ZN','INDUS','AGE','LSTAT']]
# plot histogram of all variables which have missing values
# set the number of bins to 20
# set the figure size using 'figsize'
df_missing_values.hist(bins = 20, figsize = (15,8))
# adjust the subplots
plt.tight_layout()
# display the plot
plt.show()

In [ ]:
df["CRIM"]=df["CRIM"].fillna(df["CRIM"].median())
df["ZN"]=df["ZN"].fillna(df["ZN"].median())
df["INDUS"]=df["INDUS"].fillna(df["INDUS"].median())
df["RIVER"]=df["RIVER"].fillna(df["RIVER"].mode()[0])
df["AGE"]=df["AGE"].fillna(df["AGE"].median())
df["LSTAT"]=df["LSTAT"].fillna(df["LSTAT"].mean())

# Outlier Visualization

In [ ]:
plt.figure(figsize=(15, 10))

df.boxplot()
plt.xticks(rotation=45)
plt.title("Boxplot of All Features")
plt.show()

# Correlation Heatmap

In [ ]:
# set dimensions for the plot figure
fig_dims = (20,10)
fig, ax = plt.subplots(figsize=fig_dims)

# plot the heat map
# corr: give the correlation matrix
# annot: prints the correlation values in the chart
# annot_kws: sets the font size of the annotation
sns.heatmap(df.corr(), annot = True, annot_kws = {"size": 10})


# set text size using 'fontsize'
plt.title("Correlation Heatmap Of Housing Features",fontsize = 16)
plt.xticks(rotation = 'horizontal', fontsize = 15)

# display the plot
plt.show()

# Data Type Conversion and Encoding

In [ ]:
df["RIVER"]=df["RIVER"].astype("int64")
df["RAD"]=df["RAD"].astype("object")
df.info()

In [ ]:
# use 'drop_first' to create (n-1) dummy variables
# use 'prefix' to add prefix to dummy variable
df_encoded = pd.get_dummies(df,columns=['RAD'], drop_first=True, dtype = "int64")

# Selecting Feature and target Variable

In [ ]:
X = df_encoded.drop(["PRICE"],axis = 1)
y = df_encoded["PRICE"]

# Train Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size = 0.2, random_state = 100)

In [ ]:
# to estimate the regression coefficient , a constant term of '1' needs to be added as a separate column
# 'sm.add_constant' adds the intercept to the model
X_train_ols = sm.add_constant(X_train)
X_train_ols = X_train_ols.astype("float")

X_test_ols = sm.add_constant(X_test)
X_test_ols = X_test_ols.astype("float")

# build a model with an intercept
ols_model = sm.OLS(y_train, X_train_ols).fit()
print(ols_model.summary())

In [ ]:
# Level of Errors for full model (using OLS Estiamtion)
# predict the values of target variable using train data
y_train_pred = ols_model.predict(X_train_ols)
mse_train = mean_squared_error(y_train, y_train_pred)
print ("MSE for Train Data:",mse_train)

y_test_pred = ols_model.predict(X_test_ols)
mse_test = mean_squared_error(y_test, y_test_pred)
print ("MSE for Test Data:",mse_test)

# Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# Cross Validation

In [ ]:
# 5-Fold Cross Validation

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Linear Regression

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train_scaled, y_train)

# predict the values of target variable using train data
linear_train_pred = linear_model.predict(X_train_scaled)

linear_test_pred = linear_model.predict(X_test_scaled)

In [ ]:
mse_train = mean_squared_error(y_train, linear_train_pred)
print ("MSE for Train Data:",mse_train)

mse_test = mean_squared_error(y_test, linear_test_pred)
print ("MSE for Test Data:",mse_test)

In [ ]:
# Model Summary:
print('model intercept :', linear_model.intercept_)
print('model coefficients : ', linear_model.coef_)
print('Model R-square_train : ', r2_score(y_train,linear_train_pred))
print('Model R-square_test : ', r2_score(y_test,linear_test_pred))

In [ ]:
# obtain the R-squared
r_square = linear_model.score(X_train_scaled, y_train)

# print the R-squared 
print("The R-squared value is", r_square)

# Gradient Descent + GridSearchCV

In [ ]:
# Full model using Full batch Gradient Descent (Hyper parameter tunning):

# Instantiate the SGDRegressor

sgd = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None,learning_rate='constant', random_state=42)

#Define the parameter grid, eta0 is the learning rate in Learning rate in scikit-learn implementation

param_grid = {'eta0': np.linspace(0.001, 1,1000)}

#Grid Search
sgd_grid = GridSearchCV(estimator=sgd,param_grid=param_grid,scoring='neg_mean_squared_error',cv=kf,n_jobs=-1)

#Fit grid search
sgd_grid.fit(X_train_scaled, y_train)

#Show best params
print("Best parameters found:", sgd_grid.best_params_)

# Evaluate on test set
sgd_best_model = sgd_grid.best_estimator_

In [ ]:
# Full model using Stochastic Gradient Descent

# Make predictions on your train data
sgd_train_pred = sgd_best_model.predict(X_train_scaled)

mse_train = mean_squared_error(y_train, sgd_train_pred)
print ("MSE for Train Data:",mse_train)

# Make predictions on your test data
sgd_test_pred = sgd_best_model.predict(X_test_scaled)

mse_test = mean_squared_error(y_test, sgd_test_pred)
print ("MSE for Test Data:",mse_test)

In [ ]:
# obtain the R-squared
r_square = sgd_best_model.score(X_train_scaled, y_train)

# print the R-squared 
print("The R-squared value is", r_square)

# Ridge Regression + GridSearchCV

In [ ]:
# 'alpha' assigns the regularization strength to the model
tuned_paramaters = [{'alpha':np.linspace(0.001,1,1000)}]
 
# instantiate the Ridge() method
ridge = Ridge()

# use GridSearchCV() to find the optimal value of alpha
# estimator: pass the ridge regression model
# param_grid: pass the list 'tuned_parameters'

ridge_grid = GridSearchCV(estimator = ridge, 
                          param_grid = tuned_paramaters, 
                          cv = kf, scoring = 'neg_mean_squared_error',n_jobs = -1)

# fit the model on X_train_scaled and y_train using fit()
ridge_grid.fit(X_train_scaled, y_train) # best model selected

# get the best parameters
print('Best parameters for Ridge Regression:', ridge_grid.best_params_)

ridge_best_model = ridge_grid.best_estimator_

In [ ]:
# For training set:
# ridge_train_pred: prediction made by the model on the training dataset 'X_train_scaled'
# y_train: actual values ofthe target variable for the train dataset
 
# predict the output of the target variable from the train data 
ridge_train_pred = ridge_best_model.predict(X_train_scaled)

# calculate the MSE using the "mean_squared_error" function
# MSE for the train data
mse_train = mean_squared_error(y_train, ridge_train_pred)

# print the MSE for the train set
print("MSE for Train Data: ", mse_train)
    
# For testing set:
# ridge_test_pred: prediction made by the model on the test dataset 'X_test_scaled'
# y_test: actual values of the target variable for the test dataset
 
# predict the output of the target variable from the test data
ridge_test_pred = ridge_best_model.predict(X_test_scaled)

# MSE for the test data
mse_test = mean_squared_error(y_test, ridge_test_pred)

# print the MSE for the test set
print("MSE for Test Data:", mse_test)

In [ ]:
ridge_best_model.coef_

In [ ]:
# obtain the R-squared
r_square_ridge = ridge_best_model.score(X_train_scaled, y_train)

# print the R-squared 
print("The R-squared value is", r_square_ridge)

# Lasso Regression + GridSearchCV

In [ ]:
# 'alpha' assigns the regularization strength to the model
tuned_paramaters = [{'alpha':np.linspace(0.001,1,1000)}]
 
# instantiate the Lasso() method
lasso = Lasso()

# use GridSearchCV() to find the optimal value of alpha
# estimator: pass the lasso regression model
# param_grid: pass the list 'tuned_parameters'
# cv: number of folds in k-fold 
lasso_grid = GridSearchCV(estimator = lasso, 
                          param_grid = tuned_paramaters, 
                          cv = kf, scoring = 'neg_mean_squared_error',n_jobs = -1 )

# fit the model on X_train_scaled and y_train using fit()
lasso_grid.fit(X_train_scaled, y_train)

# get the best parameters
print('Best parameters for Lasso Regression:', lasso_grid.best_params_)

lasso_best_model = lasso_grid.best_estimator_

In [ ]:
# For training set:
# lasso_train_pred: prediction made by the model on the training dataset 'X_train_scaled'
# y_train: actual values ofthe target variable for the train dataset
 
# predict the output of the target variable from the train data 
lasso_train_pred = lasso_best_model.predict(X_train_scaled)

# calculate the MSE using the "mean_squared_error" function
# MSE for the train data
mse_train = mean_squared_error(y_train, lasso_train_pred)

# print the MSE for the train set
print("MSE for Train Data:", mse_train)
    
# For testing set:
# lasso_test_pred: prediction made by the model on the test dataset 'X_test'
# y_test: actual values of the target variable for the test dataset
 
# predict the output of the target variable from the test data
lasso_test_pred = lasso_best_model.predict(X_test_scaled)

# MSE for the test data
mse_test = mean_squared_error(y_test, lasso_test_pred)

# print the MSE for the test set
print("MSE for Test Data:", mse_test)

In [ ]:
# create a dataframe to store the variable names and their corresponding coefficient values
# pass the dictionary as data to the dataframe
# 'coef_' returns the value of each coefficient
df_lasso_coeff = pd.DataFrame(data = {'Variable': X.columns, 'Coefficient': lasso_best_model.coef_})

# print the variables having the coefficient equal to zero
# 'to_list()' converts the output to the list type
print('Insignificant variables obtained from Lasso Regression when alpha is tuned:',
      df_lasso_coeff.Variable[df_lasso_coeff.Coefficient == 0].to_list())

In [ ]:
lasso_best_model.coef_

In [ ]:
# obtain the R-squared
r_square_lasso = lasso_best_model.score(X_train_scaled, y_train)

# print the R-squared 
print("The R-squared value is", r_square_lasso)

# Elastic Net Regression + GridSearchCV

In [ ]:
# create a dictionary with hyperparameters and its values
tuned_paramaters = [{'alpha':np.linspace(0.001,1,1000),
                      'l1_ratio': np.linspace(0.1,0.9,5)}]
 
# instantiate the ElasticNet() method
enet = ElasticNet()

# using GridSearchCV to find the optimal value of alpha and l1_ratio
# estimator: pass the elastic net regression model
# param_grid: pass the list 'tuned_parameters'
# cv: number of folds in k-fold 
enet_grid = GridSearchCV(estimator = enet, 
                          param_grid = tuned_paramaters, 
                          cv = kf, scoring = 'neg_mean_squared_error',n_jobs = -1 )

# fit the model on X_train_scaled and y_train
enet_grid.fit(X_train_scaled, y_train)

# get the best parameters
print('Best parameters for ELastic-net Regression: ', enet_grid.best_params_, '\n')

enet_best_model = enet_grid.best_estimator_

In [ ]:
# predict the values of target variable using test data
enet_test_pred = enet_best_model.predict(X_test_scaled)

# For training set:
# enet_train_pred: prediction made by the model on the training dataset 'X_train_scaled'
# y_train: actual values ofthe target variable for the train dataset
 
# predict the output of the target variable from the train data 
enet_train_pred = enet_best_model.predict(X_train_scaled)

# calculate the MSE using the "mean_squared_error" function
# MSE for the train data
mse_train = mean_squared_error(y_train, enet_train_pred)

# print the MSE for the train set
print("MSE for Train Data:", mse_train)
    
# MSE for the test data
mse_test = mean_squared_error(y_test, enet_test_pred)

# print the MSE for the test set
print("MSE for Test Data:", mse_test)

In [ ]:
enet_best_model.coef_

In [ ]:
# obtain the R-squared
r_square_enet = enet_best_model.score(X_train_scaled, y_train)

# print the R-squared 
print("The R-squared value is", r_square_enet)